# Tutorial Completo: Nuevas Características de la API de OpenAI GPT-5

El modelo **GPT-5** introduce capacidades de razonamiento profundo, control fino de verbosidad, ejecución de herramientas con entrada de texto libre (*freeform*), gramáticas libres de contexto (CFG), subconjuntos de herramientas permitidas (*allowed tools*) y explicaciones previas a herramientas (*preambles*).

Este notebook interactivo permite recorrer cada una de las 7 características clave en código paso a paso usando la nueva **Responses API** (`client.responses.create`).

---

## 0. Configuración Inicial y Autenticación
Cargamos las variables de entorno desde el archivo `.env` e inicializamos el cliente oficial de OpenAI.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('Cliente inicializado correctamente.')

Cliente inicializado correctamente.


## 1. Nivel de Razonamiento (`Reasoning Effort`)
Permite controlar cuánto 'tiempo de pensamiento' interno invierte el modelo antes de responder: `minimal`, `low`, `medium` o `high`.
- `minimal`: Ideal para respuestas rápidas, comandos o código simple donde se prioriza latencia mínima y cero tokens de razonamiento.
- `high`: Ideal para deducciones complejas, resolución de algoritmos y tareas críticas.

In [2]:
# 1.1 Razonamiento Minimal
resp_min = client.responses.create(
    model='gpt-5',
    input='¿Quién es el creador de Python y en qué año se lanzó?',
    reasoning={'effort': 'minimal'},
)
print('--- MINIMAL EFFORT ---')
print(resp_min.output_text)
print(f'Reasoning tokens: {resp_min.usage.output_tokens_details.reasoning_tokens}')
print(f'Total tokens: {resp_min.usage.total_tokens}')

--- MINIMAL EFFORT ---
El creador de Python es Guido van Rossum, y la primera versión pública se lanzó en 1991.
Reasoning tokens: 0
Total tokens: 62


In [3]:
# 1.2 Razonamiento High
resp_high = client.responses.create(
    model='gpt-5',
    input='¿Quién es el creador de Python y en qué año se lanzó?',
    reasoning={'effort': 'high'},
)
print('--- HIGH EFFORT ---')
print(resp_high.output_text)
print(f'Reasoning tokens: {resp_high.usage.output_tokens_details.reasoning_tokens}')
print(f'Total tokens: {resp_high.usage.total_tokens}')

--- HIGH EFFORT ---
Python fue creado por Guido van Rossum y su primera versión pública se lanzó en 1991.
Reasoning tokens: 256
Total tokens: 345


## 2. Nivel de Verbosidad (`Verbosity`)
Controla la longitud y el detalle de la respuesta generada sin necesidad de ajustar el prompt: `low`, `medium` o `high`.

In [4]:
# Verbosidad baja (concisa)
resp_low = client.responses.create(
    model='gpt-5',
    input='¿Quién es Guido van Rossum?',
    reasoning={'effort': 'minimal'},
    text={'verbosity': 'low'},
)
print('Low verbosity:')
print(resp_low.output_text)

# Verbosidad alta (detallada)
resp_high_v = client.responses.create(
    model='gpt-5',
    input='¿Quién es Guido van Rossum?',
    reasoning={'effort': 'minimal'},
    text={'verbosity': 'high'},
)
print('\nHigh verbosity:')
print(resp_high_v.output_text)

Low verbosity:
Guido van Rossum es un programador neerlandés, creador del lenguaje de programación Python. Inició Python a finales de los 80 y lo lanzó públicamente en 1991. Fue el “Benevolent Dictator For Life” (BDFL) de Python hasta 2018, guiando su desarrollo. Ha trabajado en instituciones como CWI, CNRI, Google y Dropbox, y desde 2020 se unió a Microsoft para seguir impulsando Python. Es reconocido por diseñar un lenguaje legible, sencillo y con una comunidad muy activa.

High verbosity:
Guido van Rossum es un programador neerlandés conocido por ser el creador del lenguaje de programación Python. Nació el 31 de enero de 1956 en los Países Bajos. Comenzó a desarrollar Python a finales de los años 80 y publicó su primera versión en 1991 mientras trabajaba en el CWI (Centrum Wiskunde & Informatica) en Á Ámsterdam. Diseñó Python con énfasis en la legibilidad, la simplicidad y una sintaxis clara, inspirándose en parte en ABC, un lenguaje en el que también había trabajado.

Algunos punto

## 3. Conversaciones Multi-Turno con Traspaso de Cadena de Pensamiento (`CoT Handoff`)
En la Responses API, el parámetro `previous_response_id` preserva la cadena de pensamiento interna del turno anterior, evitando repetición de cálculos y manteniendo un contexto coherente.

In [5]:
first = client.responses.create(
    model='gpt-5',
    input='¿Quién es el padre de Python?',
    reasoning={'effort': 'minimal'},
    text={'verbosity': 'low'},
)
print(f'Turno 1 [ID: {first.id}]:')
print(first.output_text)

followup = client.responses.create(
    model='gpt-5',
    previous_response_id=first.id,
    input='Escribe un párrafo para blog resumiendo su impacto tecnológico.',
    reasoning={'effort': 'medium'},
    text={'verbosity': 'high'},
)
print('\nTurno 2 (con CoT Handoff):')
print(followup.output_text)

Turno 1 [ID: resp_07c55ab0334755a8006ad984d2598487d2adfbbac01c50ce28]:
Guido van Rossum.

Turno 2 (con CoT Handoff):
Guido van Rossum, “padre de Python”, transformó la industria al diseñar un lenguaje centrado en la legibilidad, la simplicidad y el pragmatismo (“batteries included”) que redujo drásticamente la barrera de entrada y se volvió la lengua franca desde la web hasta la ciencia de datos y la IA. Como líder del proyecto durante casi tres décadas, impulsó un modelo de mejora abierta (PEPs), fomentó una cultura comunitaria ejemplar y guio transiciones clave como el paso a Python 3 y la adopción del tipado gradual, mientras el ecosistema (PyPI, pip, Jupyter) se consolidaba. Su huella se ve en la educación, en la automatización y en plataformas a escala como YouTube, Instagram o Dropbox. Tras etapas en Google y Dropbox, se unió a Microsoft para ayudar a acelerar el intérprete, manteniendo a Python competitivo sin sacrificar su filosofía. Incluso al ceder el rol de BDFL en 2018 y da

## 4. Entrada Libre en Herramientas (`Freeform Input`)
Las herramientas personalizadas (`type: custom`) en GPT-5 pueden recibir texto crudo, scripts o consultas SQL directamente mediante el atributo `input`, sin obligar al modelo a serializar en JSON.

In [6]:
import random

def run_sql_query(sql: str) -> str:
    cats = ['Electronics', 'Clothing', 'Furniture', 'Toys', 'Books']
    res = 'category   | total_sales\n' + '-' * 28 + '\n'
    for c in cats:
        res += f'{c:<11} | {random.randint(5000, 200000)}\n'
    return res

tools = [{
    'type': 'custom',
    'name': 'sql_query_runner',
    'description': 'Runs raw SQL queries on the company sales database.',
}]

msgs = [{'role': 'user', 'content': 'Show me the total sales for each product category last month.'}]

resp = client.responses.create(model='gpt-5', tools=tools, input=msgs)
msgs += resp.output

tool_call = next(x for x in resp.output if getattr(x, 'type', '') in ('custom_tool_call', 'function_call', 'tool_call'))
sql_text = (getattr(tool_call, 'input', None) or getattr(tool_call, 'arguments', '')).strip()
print('SQL generado por GPT-5:')
print(sql_text)

tool_result = run_sql_query(sql_text)
msgs.append({
    'type': 'function_call_output',
    'call_id': tool_call.call_id,
    'output': tool_result,
})

final = client.responses.create(model='gpt-5', tools=tools, input=msgs)
print('\nRespuesta final:')
print(final.output_text)

SQL generado por GPT-5:
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'
  AND table_schema NOT IN ('information_schema', 'pg_catalog')
ORDER BY table_schema, table_name;

Respuesta final:
Total sales by category for September 2026:
- Toys: 175,407
- Books: 85,534
- Furniture: 75,787
- Clothing: 73,301
- Electronics: 31,171

Want this broken down by week or compared to the prior month?


## 5. Salidas Restringidas por Gramáticas CFG (`Lark Grammar`)
Permite imponer reglas gramaticales estrictas mediante gramáticas libres de contexto para garantizar que las consultas generadas sigan una sintaxis exacta.

In [7]:
lark_grammar = r'''
start: SELECT CATEGORY COMMA SUM LPAREN SALES RPAREN AS TOTAL_SALES FROM ORDERS WHERE ORDER_MONTH EQ ESCAPED_STRING GROUP BY CATEGORY ORDER BY TOTAL_SALES (DESC|ASC)?

SELECT: "SELECT"
CATEGORY: "category"
COMMA: ","
SUM: "SUM"
LPAREN: "("
SALES: "sales"
RPAREN: ")"
AS: "AS"
TOTAL_SALES: "total_sales"
FROM: "FROM"
ORDERS: "orders"
WHERE: "WHERE"
ORDER_MONTH: "order_month"
EQ: "="
GROUP: "GROUP"
BY: "BY"
ORDER: "ORDER"
DESC: "DESC"
ASC: "ASC"

%import common.ESCAPED_STRING
%ignore /[ \t\r\n]+/
'''

grammar_tools = [{
    'type': 'custom',
    'name': 'sql_query_runner',
    'description': 'Runs raw SQL on the sales DB.',
    'format': {
        'type': 'grammar',
        'syntax': 'lark',
        'definition': lark_grammar,
    },
}]

g_msgs = [{'role': 'user', 'content': 'Show me the total sales for each product category last month.'}]
g_resp = client.responses.create(model='gpt-5', input=g_msgs, tools=grammar_tools, text={'format': {'type': 'text'}})
g_msgs += g_resp.output
g_tool_call = next(x for x in g_resp.output if getattr(x, 'type', '') in ('custom_tool_call', 'function_call', 'tool_call'))
g_sql = (getattr(g_tool_call, 'input', None) or getattr(g_tool_call, 'arguments', '')).strip()
print('SQL conforme a gramática Lark:')
print(g_sql)

SQL conforme a gramática Lark:
SELECT category, SUM(sales) AS total_sales FROM orders WHERE order_month = "2026-09" GROUP BY category ORDER BY total_sales DESC


## 6. Subconjunto de Herramientas Permitidas (`Allowed Tools`)
Permite definir un catálogo amplio de herramientas pero limitar al modelo a usar un subconjunto específico según la etapa del flujo con `mode: 'auto'` o `mode: 'required'`.

In [8]:
all_tools = [
    {
        'type': 'function',
        'name': 'get_weather',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
    {
        'type': 'function',
        'name': 'send_email',
        'parameters': {
            'type': 'object',
            'properties': {'to': {'type': 'string'}, 'body': {'type': 'string'}},
            'required': ['to', 'body'],
        },
    },
]

at_resp = client.responses.create(
    model='gpt-5',
    input="What's the weather in Oslo?",
    tools=all_tools,
    tool_choice={
        'type': 'allowed_tools',
        'mode': 'required',
        'tools': [{'type': 'function', 'name': 'get_weather'}],
    },
)
for item in at_resp.output:
    if getattr(item, 'type', None) in ('function_call', 'tool_call', 'custom_tool_call'):
        print('Herramienta ejecutada de forma forzada:', getattr(item, 'name', None))
        print('Argumentos:', getattr(item, 'arguments', None))

Herramienta ejecutada de forma forzada: get_weather
Argumentos: {"city":"Oslo"}


## 7. Preámbulos en Ejecución de Herramientas (`Preambles`)
Genera una explicación breve y visible para el usuario antes de invocar la herramienta, facilitando la transparencia y la depuración de agentes autónomos.

In [9]:
import json

pre_tools = [{
    'type': 'function',
    'name': 'get_weather',
    'description': 'Get current temperature for a city.',
    'parameters': {
        'type': 'object',
        'properties': {'city': {'type': 'string'}},
        'required': ['city'],
        'additionalProperties': False
    },
    'strict': True,
}]

pre_msgs = [
    {'role': 'system', 'content': "Before you call a tool, explain why you are calling it in ONE short sentence prefixed with 'Preamble:'."},
    {'role': 'user', 'content': "What's the weather in Oslo?"}
]

pre_resp = client.responses.create(model='gpt-5', input=pre_msgs, tools=pre_tools)
for item in pre_resp.output:
    if getattr(item, 'type', None) == 'message':
        print('PREÁMBULO VISIBLE:', getattr(item, 'content', None))
    if getattr(item, 'type', None) in ('function_call', 'tool_call', 'custom_tool_call'):
        print('TOOL INVOCADO:', getattr(item, 'name', None))
        print('ARGUMENTOS:', getattr(item, 'arguments', None))

PREÁMBULO VISIBLE: [ResponseOutputText(annotations=[], text='Preamble: I’m going to look up the current temperature in Oslo.', type='output_text', logprobs=[])]
TOOL INVOCADO: get_weather
ARGUMENTOS: { "city": "Oslo" }
